In [1]:
!pip install lightgbm -q


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/ai-camp-2-03-1/covertype-classification/sample_submission.csv
/kaggle/input/competitions/ai-camp-2-03-1/covertype-classification/val.csv
/kaggle/input/competitions/ai-camp-2-03-1/covertype-classification/train.csv
/kaggle/input/competitions/ai-camp-2-03-1/covertype-classification/test.csv
/kaggle/input/competitions/ai-camp-2-03-1/covertype-classification/label_map.json


In [3]:
import pandas as pd, numpy as np
from sklearn.metrics import accuracy_score
import lightgbm as lgb

# ─── Feature Engineering ───────────────────────────────────────────
def engineer_features(df):
    d = df.copy()
    d['euc_hydro']  = np.sqrt(d['horizontal_distance_to_hydrology']**2 + d['vertical_distance_to_hydrology']**2)
    d['mean_hill']  = (d['hillshade_9am'] + d['hillshade_noon'] + d['hillshade_3pm']) / 3
    d['hill_range'] = d['hillshade_9am'] - d['hillshade_3pm']
    d['elev_road']  = d['elevation'] - d['horizontal_distance_to_roadways']
    d['elev_hydro'] = d['elevation'] - d['horizontal_distance_to_hydrology']
    d['elev_fire']  = d['elevation'] - d['horizontal_distance_to_fire_points']
    for c in ['horizontal_distance_to_hydrology','horizontal_distance_to_roadways','horizontal_distance_to_fire_points']:
        d[f'log_{c}'] = np.log1p(d[c].clip(0))
    d['asp_sin']  = np.sin(np.radians(d['aspect']))
    d['asp_cos']  = np.cos(np.radians(d['aspect']))
    sc = [c for c in d.columns if c.startswith('soil_type_id_')]
    d['soil_sum'] = d[sc].sum(axis=1)
    wc = [c for c in d.columns if c.startswith('wilderness_area_id_')]
    d['wild_idx'] = d[wc].values.argmax(axis=1)
    d['rd_fire']  = abs(d['horizontal_distance_to_roadways'] - d['horizontal_distance_to_fire_points'])
    d['rd_hydro'] = abs(d['horizontal_distance_to_roadways'] - d['horizontal_distance_to_hydrology'])
    return d

DATA = "/kaggle/input/competitions/ai-camp-2-03-1/covertype-classification"

train = engineer_features(pd.read_csv(f"{DATA}/train.csv"))
val   = engineer_features(pd.read_csv(f"{DATA}/val.csv"))
test  = engineer_features(pd.read_csv(f"{DATA}/test.csv"))

feat_cols = [c for c in train.columns if c not in ['id','label_id']]
X_tr, y_tr = train[feat_cols].values, train['label_id'].values
X_va, y_va = val[feat_cols].values, val['label_id'].values
X_te       = test[feat_cols].values
X_full = np.vstack([X_tr, X_va]); y_full = np.concatenate([y_tr, y_va])

# ─── LightGBM (best model for this type of data) ───────────────────
params = {
    'objective':'multiclass','num_class':7,'metric':'multi_logloss',
    'learning_rate':0.05,'num_leaves':255,'max_depth':10,
    'min_child_samples':20,'feature_fraction':0.8,
    'bagging_fraction':0.8,'bagging_freq':5,
    'reg_alpha':0.1,'reg_lambda':0.1,'n_jobs':-1,'verbose':-1,'seed':42,
}

ds_tr = lgb.Dataset(X_tr, label=y_tr)
ds_va = lgb.Dataset(X_va, label=y_va, reference=ds_tr)
cb = [lgb.early_stopping(100,verbose=False), lgb.log_evaluation(100)]
booster = lgb.train(params, ds_tr, num_boost_round=3000, valid_sets=[ds_va], callbacks=cb)
best_iter = booster.best_iteration
acc = accuracy_score(y_va, booster.predict(X_va).argmax(axis=1))
print(f"Val Accuracy: {acc:.4f}")  # Expect ~97%+

# ─── Retrain on train+val, predict test ────────────────────────────
ds_full = lgb.Dataset(X_full, label=y_full)
booster_full = lgb.train(params, ds_full, num_boost_round=best_iter)
test_preds = booster_full.predict(X_te).argmax(axis=1)

pd.DataFrame({'id': test['id'], 'label_id': test_preds}).to_csv("submission.csv", index=False)

[100]	valid_0's multi_logloss: 0.275063


[200]	valid_0's multi_logloss: 0.193911


[300]	valid_0's multi_logloss: 0.148745


[400]	valid_0's multi_logloss: 0.121655


[500]	valid_0's multi_logloss: 0.106043


...

[1600]	valid_0's multi_logloss: 0.070824


[1700]	valid_0's multi_logloss: 0.0706539


[1800]	valid_0's multi_logloss: 0.0705677


[1900]	valid_0's multi_logloss: 0.0705235


Val Accuracy: 0.9741
